# Hyperparameters Tuning NOTEBOOK

Utiliza o framework Optuna para otimizar hiperparâmetros de treino. Neste caso, apenas o utilizaremos para encontrar valores ótimos de Learning Rate.
Realizou-se a otimização para cada uma da três arquiteturas de deep learning: Unet, AttentionUnet e UNETR, e para cada um dos datasets: MoNuSeg, PanNuke e NuInSeg.

In [24]:
import torch
from monai.data import Dataset, DataLoader 
from monai.networks.nets import UNet, AttentionUnet, UNETR
from monai.transforms import Compose, LoadImaged, ScaleIntensityd, RandFlipd, ToTensord, Activations, AsDiscrete, EnsureChannelFirstd
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
import pandas as pd
import os
from pathlib import Path
import optuna
from functools import partial

Abaixo estão definidas funções para organizar os dados de entrada do MONAI

In [25]:
CWD = Path(os.getcwd()).parent
print(CWD)

def prepare_splits(labels_csv_path):
    labels_df = pd.read_csv(labels_csv_path)
    train_df = labels_df[labels_df['split'] == 'train']
    val_df = labels_df[labels_df['split'] == 'val']
    test_df = labels_df[labels_df['split'] == 'test']
    return train_df, val_df, test_df

def get_dataset_paths(dataset_name):
    CWD = Path(os.getcwd()).parent
    valid_datasets = ['NuInSeg', 'PanNuke', 'MoNuSeg']
    if dataset_name not in valid_datasets:
        raise ValueError(f"Invalid dataset name: {dataset_name}. Valid options are: {valid_datasets}")
    image_path = Path(CWD / f'data/processed/{dataset_name}/Images/')
    mask_path = Path(CWD / f'data/processed/{dataset_name}/Masks/')
    return image_path, mask_path

def prepare_datasets(df, dataset_name):
    image_path, mask_path = get_dataset_paths(dataset_name)
    print(image_path)
    images = [str(image_path / f"{row['FileName']}.npy") for _, row in df.iterrows()]
    masks = [str(mask_path / f"{row['FileName']}.npy") for _, row in df.iterrows()]
    monai_dict = [{'image': image, 'label': mask} for image, mask in zip(images, masks)]
    return monai_dict

/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos


In [26]:
MoNuSeg_csv = CWD / 'data/processed/MoNuSeg/labels.csv'
PanNuke_csv = CWD / 'data/processed/PanNuke/labels.csv'
NuInSeg_csv = CWD / 'data/processed/NuInSeg/labels.csv'

MoNuSeg_train_df, MoNuSeg_val_df, MoNuSeg_test_df = prepare_splits(MoNuSeg_csv)
PanNuke_train_df, PanNuke_val_df, PanNuke_test_df = prepare_splits(PanNuke_csv)
NuInSeg_train_df, NuInSeg_val_df, NuInSeg_test_df = prepare_splits(NuInSeg_csv)

MoNuSeg_train_data = prepare_datasets(MoNuSeg_train_df, 'MoNuSeg')
PanNuke_train_data = prepare_datasets(PanNuke_train_df, 'PanNuke')
NuInSeg_train_data = prepare_datasets(NuInSeg_train_df, 'NuInSeg')

MoNuSeg_val_data = prepare_datasets(MoNuSeg_val_df, 'MoNuSeg')
PanNuke_val_data = prepare_datasets(PanNuke_val_df, 'PanNuke')
NuInSeg_val_data = prepare_datasets(NuInSeg_val_df, 'NuInSeg')


transforms_train = Compose([
    LoadImaged(keys=['image', 'label'], reader="NumpyReader"),

    EnsureChannelFirstd(keys=['image'], channel_dim=-1),
    EnsureChannelFirstd(keys=['label'], channel_dim="no_channel"),

    ScaleIntensityd(keys=['image']),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
    ToTensord(keys=['image', 'label'])
    ])

transforms_val = Compose([
    LoadImaged(keys=['image', 'label'], reader="NumpyReader"),
    EnsureChannelFirstd(keys=['image'], channel_dim=-1),
    EnsureChannelFirstd(keys=['label'], channel_dim="no_channel"),
    ScaleIntensityd(keys=['image']),
    ToTensord(keys=['image', 'label'])
    ])


MoNuSeg_train_ds = Dataset(data=MoNuSeg_train_data, transform=transforms_train)
PanNuke_train_ds = Dataset(data=PanNuke_train_data, transform=transforms_train)
NuInSeg_train_ds = Dataset(data=NuInSeg_train_data, transform=transforms_train)

MoNuSeg_val_ds = Dataset(data=MoNuSeg_val_data, transform=transforms_val)
PanNuke_val_ds = Dataset(data=PanNuke_val_data, transform=transforms_val)
NuInSeg_val_ds = Dataset(data=NuInSeg_val_data, transform=transforms_val)

datasets = {
    'MoNuSeg': {
        'train': MoNuSeg_train_ds,
        'val': MoNuSeg_val_ds
    },
    'PanNuke': {
        'train': PanNuke_train_ds,
        'val': PanNuke_val_ds
    },
    'NuInSeg': {    
        'train': NuInSeg_train_ds,
        'val': NuInSeg_val_ds
        }
}        


/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/MoNuSeg/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/PanNuke/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/NuInSeg/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/MoNuSeg/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/PanNuke/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/NuInSeg/Images


Definição do objetivo do optuna: maximizar o DICE score

In [27]:
post_pred = Compose([Activations(keys="pred", sigmoid=True), AsDiscrete(keys="pred", threshold=0.5)])
post_label = Compose([AsDiscrete(threshold=0.5)])
def objective(trial, dataset_name, model_name):

    train_ds = datasets[dataset_name]['train']
    val_ds = datasets[dataset_name]['val']
    
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam"])
    batch_size = trial.suggest_categorical("batch_size", [16])
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if model_name == "UNet":
        model = UNet( # unet do monai
            spatial_dims=2,
            in_channels=3,
            out_channels=1,
            channels=(16, 32, 64, 128, 256),
            strides=(2, 2, 2, 2),
            num_res_units=2,
        ).to(device)
        
    elif model_name == "AttentionUnet":
        model = AttentionUnet( # attention unet do monai
            spatial_dims=2,
            in_channels=3,
            out_channels=1,
            channels=(16, 32, 64, 128, 256),
            strides=(2, 2, 2, 2)
        ).to(device)
        
    elif model_name == "UNETR":
        model = UNETR( # unetr do monai
            in_channels=3,
            out_channels=1,
            img_size=(256, 256),
            spatial_dims=2
        ).to(device)


    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=1, num_workers=2)
    

    loss_function = DiceCELoss(sigmoid=True)
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
        
    dice_metric = DiceMetric(include_background=True, reduction="mean")
    
    max_epochs = 15 
    best_metric = -1
    

    for epoch in range(max_epochs):
            model.train()
            for batch_data in train_loader:
                inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = loss_function(outputs, labels)
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.no_grad():
                for val_data in val_loader:
                    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                    val_outputs = model(val_inputs)

                    val_outputs = [post_pred(i) for i in val_outputs]
                    val_labels = [post_label(i) for i in val_labels]
                    
                    dice_metric(y_pred=val_outputs, y=val_labels)
                    
                metric = dice_metric.aggregate().item()
                dice_metric.reset()
                
                if metric > best_metric:
                    best_metric = metric
                    
    return best_metric

Loop duplo para encontrar as learning rates para cada combinação arquitetura-dataset

In [ ]:
dataset_names = ["MoNuSeg", "PanNuke", "NuInSeg"]
model_names = ["UNet", "AttentionUnet", "UNETR"]

resultados_finais = {}
arquivo_saida = "relatorio_optuna.txt"

with open(arquivo_saida, "w", encoding="utf-8") as f:
    f.write("INICIANDO OTIMIZAÇÃO DE HIPERPARÂMETROS\n")

for dataset_name in dataset_names:
    resultados_finais[dataset_name] = {}
    
    for model_name in model_names:
        cabecalho = f"\n" + "="*60 + f"\n OTIMIZANDO: Modelo {model_name} | Dataset {dataset_name}\n" + "="*60
        print(cabecalho)
        
        study_name = f"Opt_{model_name}_{dataset_name}"
        study = optuna.create_study(direction="maximize", study_name=study_name)
        
        objective_fn = partial(objective, dataset_name=dataset_name, model_name=model_name)
        
        study.optimize(objective_fn, n_trials=15)
        
        resultados_finais[dataset_name][model_name] = study.best_trial
        
        resultado_parcial = f"\n--- Concluído: {model_name} no {dataset_name} ---\n"
        resultado_parcial += f"Melhor Dice Score: {study.best_trial.value:.4f}\n"
        resultado_parcial += "Melhores Hiperparâmetros encontrados:\n"
        for key, value in study.best_trial.params.items():
            resultado_parcial += f"    {key}: {value}\n"
            
        print(resultado_parcial)
        
        with open(arquivo_saida, "a", encoding="utf-8") as f:
            f.write(cabecalho + "\n")
            f.write(resultado_parcial)


# ==========================================
# RELATÓRIO FINAL
# ==========================================

for dataset_name in dataset_names:
    texto_final += f"\n>>> DATASET: {dataset_name}\n"
    for model_name in model_names:
        best_trial = resultados_finais[dataset_name][model_name]
        texto_final += f"  - {model_name.ljust(15)} | Melhor Dice: {best_trial.value:.4f} | Params: {best_trial.params}\n"

[I 2026-06-21 22:56:38,264] A new study created in memory with name: Opt_UNet_MoNuSeg



 OTIMIZANDO: Modelo UNet | Dataset MoNuSeg


[I 2026-06-21 22:56:49,874] Trial 0 finished with value: 0.7748160362243652 and parameters: {'lr': 0.00011669952595321336, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.7748160362243652.
[I 2026-06-21 22:57:01,435] Trial 1 finished with value: 0.6422098278999329 and parameters: {'lr': 0.00011963893806176947, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.7748160362243652.
[I 2026-06-21 22:57:13,058] Trial 2 finished with value: 0.7889332175254822 and parameters: {'lr': 0.0005014318939557268, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 2 with value: 0.7889332175254822.
[I 2026-06-21 22:57:24,699] Trial 3 finished with value: 0.7909520864486694 and parameters: {'lr': 0.0009409831297948334, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 3 with value: 0.7909520864486694.
[I 2026-06-21 22:57:36,301] Trial 4 finished with value: 0.7984301447868347 and parameters: {'lr': 0.0004495217928876565, 'optimizer': 'Adam', 'batch_size': 1


--- Concluído: UNet no MoNuSeg ---
Melhor Dice Score: 0.8052
Melhores Hiperparâmetros encontrados:
    lr: 0.0009945819807331581
    optimizer: Adam
    batch_size: 16

 OTIMIZANDO: Modelo AttentionUnet | Dataset MoNuSeg


[I 2026-06-21 22:59:51,232] Trial 0 finished with value: 0.8099464774131775 and parameters: {'lr': 0.0005163938409286957, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.8099464774131775.
[I 2026-06-21 23:00:12,463] Trial 1 finished with value: 0.7959645986557007 and parameters: {'lr': 0.00024962831671495814, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.8099464774131775.
[I 2026-06-21 23:00:33,751] Trial 2 finished with value: 0.806577742099762 and parameters: {'lr': 0.00044795534779857677, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.8099464774131775.
[I 2026-06-21 23:00:55,002] Trial 3 finished with value: 0.7520505785942078 and parameters: {'lr': 0.00011130604697357258, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.8099464774131775.
[I 2026-06-21 23:01:16,189] Trial 4 finished with value: 0.8160930275917053 and parameters: {'lr': 0.000796325412507977, 'optimizer': 'Adam', 'batch_size': 16


--- Concluído: AttentionUnet no MoNuSeg ---
Melhor Dice Score: 0.8161
Melhores Hiperparâmetros encontrados:
    lr: 0.000796325412507977
    optimizer: Adam
    batch_size: 16

 OTIMIZANDO: Modelo UNETR | Dataset MoNuSeg


[I 2026-06-21 23:05:56,843] Trial 0 finished with value: 0.7671790719032288 and parameters: {'lr': 0.0003465759385534776, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.7671790719032288.
[I 2026-06-21 23:07:04,782] Trial 1 finished with value: 0.7807344794273376 and parameters: {'lr': 0.00025163531327898294, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 1 with value: 0.7807344794273376.
[I 2026-06-21 23:08:12,987] Trial 2 finished with value: 0.7837589979171753 and parameters: {'lr': 0.000952013771124192, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 2 with value: 0.7837589979171753.
[I 2026-06-21 23:09:21,300] Trial 3 finished with value: 0.7819749712944031 and parameters: {'lr': 0.0006322959916884557, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 2 with value: 0.7837589979171753.
[I 2026-06-21 23:10:29,208] Trial 4 finished with value: 0.7793934345245361 and parameters: {'lr': 0.00019940204092490067, 'optimizer': 'Adam', 'batch_size': 16


--- Concluído: UNETR no MoNuSeg ---
Melhor Dice Score: 0.7864
Melhores Hiperparâmetros encontrados:
    lr: 0.0002108254992289144
    optimizer: Adam
    batch_size: 16

 OTIMIZANDO: Modelo UNet | Dataset PanNuke


[I 2026-06-21 23:24:48,827] Trial 0 finished with value: 0.8219161629676819 and parameters: {'lr': 0.0005492476442445547, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.8219161629676819.
[I 2026-06-21 23:27:46,728] Trial 1 finished with value: 0.8038796782493591 and parameters: {'lr': 0.00017418326322619186, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.8219161629676819.
[I 2026-06-21 23:30:43,985] Trial 2 finished with value: 0.8049323558807373 and parameters: {'lr': 0.00015434626510384993, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.8219161629676819.
[I 2026-06-21 23:33:41,286] Trial 3 finished with value: 0.8160477876663208 and parameters: {'lr': 0.00028977297720915164, 'optimizer': 'Adam', 'batch_size': 16}. Best is trial 0 with value: 0.8219161629676819.
[I 2026-06-21 23:36:38,591] Trial 4 finished with value: 0.8220736980438232 and parameters: {'lr': 0.000497093226815307, 'optimizer': 'Adam', 'batch_size': 1


--- Concluído: UNet no PanNuke ---
Melhor Dice Score: 0.8237
Melhores Hiperparâmetros encontrados:
    lr: 0.000842897310574267
    optimizer: Adam
    batch_size: 16

 OTIMIZANDO: Modelo AttentionUnet | Dataset PanNuke
